# Class 2 — Spark Structured Streaming (EMR)

This notebook explains how Spark processes **unbounded** data -- streams that never "finish" -- and how that changes the programming model, correctness guarantees, and operational concerns compared to batch.

## 1. The core idea: "streaming as an unbounded table"

Structured Streaming's foundational trick is to treat a stream as a table that keeps growing. You write the **same DataFrame/SQL transformations** you'd write for batch -- the engine repeatedly (a) reads new data since the last checkpoint, (b) applies the transformations incrementally, and (c) writes/updates the result to a sink.

## 2. Micro-batch vs continuous processing

- **Micro-batch** (the default, and what this project uses): Spark runs a small batch job every trigger interval.
- **Continuous processing** (experimental, narrow operator support): true low-latency (~1ms) processing without discrete batches. Rarely used in production.

## 3. Triggers

- `trigger(processingTime="30 seconds")` -- fixed-interval micro-batches; a true "always on" stream.
- `trigger(availableNow=True)` -- process everything currently available then **stop**. This is what production notebooks use for scheduled/batch-like streaming runs.
- `trigger(once=True)` -- legacy single micro-batch; superseded by `availableNow`.

## 4. Output modes

- **Append** -- only new result rows since the last trigger are emitted. Required for most sinks (Delta, Kafka). Works for non-aggregated data, and for aggregations *with* a watermark.
- **Update** -- only rows whose aggregate value changed are emitted.
- **Complete** -- the entire result table is re-emitted every trigger. Only viable for small aggregated results.

## 5. Event time, watermarks, and late data

Two different clocks matter: **event time** (when the event actually happened) and **processing time** (when Spark received it). If you window by event time, Spark needs a rule for when it's safe to stop waiting for late data -- that's the **watermark**: `withWatermark("event_ts", "10 minutes")` means "once I've seen an event with timestamp T, assume I will never see an event more than 10 minutes older than T."

It is a **tradeoff knob**, not a filter: too short and legitimately late events get dropped; too long and Spark keeps more state for longer, with results lagging further behind wall-clock time.

## 6. State, checkpoints, and fault tolerance

- **State** -- data Spark must remember between micro-batches, stored in the executors' state store, backed by the checkpoint location.
- **Checkpoint** -- a directory (must be durable storage -- S3, never local disk) storing source read progress, the state store, and sink commit metadata. **Never delete or share a production checkpoint** unless you intend to reset/reprocess from scratch.
- Correctness end to end also depends on the **sink** being idempotent/transactional. Delta Lake's transactional writes plus streaming checkpoint offsets are what make Kafka -> Spark -> Delta exactly-once in practice.

## 7. Stateful operations at a glance

| Operation | Needs watermark? | Why |
|---|---|---|
| `groupBy(window(...))` aggregation | Yes | Otherwise Spark keeps every window's state forever |
| `dropDuplicates(["id"])` on a stream | Yes | Otherwise Spark must remember every id ever seen |
| Stream-stream join | Yes (both sides) | Both sides buffer unmatched rows until the watermark expires them |
| Stream-static join | No | The static side has no state to expire |

Below, we build a small runnable streaming pipeline using Spark's `rate` source so these concepts can be observed without needing Kafka/MSK yet -- that's covered in `03_kafka_and_msk.ipynb`.

Run the cell below first -- it configures Delta Lake for this notebook's Spark session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "class_emr"
base_path = "s3://<your-lakehouse-bucket>/data/class-emr"  # from `terraform output lakehouse_bucket_name`

def table(name):
    return f"`{schema}`.`{name}`"

def path(*parts):
    return base_path.rstrip("/") + "/" + "/".join(p.strip("/") for p in parts)

def checkpoint(name):
    return path("checkpoints", name)

from pyspark.sql import functions as F

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema}` LOCATION '{path('tables')}'")
spark.sql(f"USE `{schema}`")
print(f"schema={schema}, base_path={base_path}")

## Step 1 — A streaming source

`rate` emits rows with a monotonically increasing `value` and a `timestamp`, at a controlled rate. We shape it into fake clickstream events so the mechanics look identical to a real Kafka payload.

In [ ]:
raw_stream = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 50)
    .option("numPartitions", 2)
    .load()
)

events_stream = raw_stream.select(
    F.concat(F.lit("evt-"), F.col("value")).alias("event_id"),
    F.col("timestamp").alias("event_ts"),
    F.concat(F.lit("u"), F.lpad((F.col("value") % 200).cast("string"), 5, "0")).alias("user_id"),
    F.concat(F.lit("p"), F.lpad(((F.col("value") % 50) + 1).cast("string"), 4, "0")).alias("product_id"),
    F.when((F.col("value") % 10) == 0, "purchase").otherwise("view").alias("event_type"),
    F.when((F.col("value") % 10) == 0, 1).otherwise(0).cast("int").alias("quantity"),
    F.when((F.col("value") % 10) == 0, 29.99).otherwise(0.0).cast("double").alias("price"),
)

print("Is streaming:", events_stream.isStreaming)

## Step 2 — Stateless streaming write (append mode, no aggregation)

The simplest streaming query: no state, no watermark needed. Every micro-batch is just filtered and appended.

In [ ]:
bronze_query = (
    events_stream
    .withColumn("_ingest_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint("class_streaming_bronze"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table("class_streaming_bronze"))
)

bronze_query.awaitTermination()
print("AvailableNow bronze streaming run completed")
spark.table(table("class_streaming_bronze")).orderBy(F.desc("event_ts")).limit(10).show(truncate=False)

Inspect the query's recent progress. `numInputRows`, `inputRowsPerSecond`, and `durationMs` are the metrics you'd wire into monitoring/alerting in production (see `emr-notebooks/05_observability_testing_performance.ipynb`).

In [ ]:
import json as _json
for p in bronze_query.recentProgress[-3:]:
    print(_json.dumps({k: p[k] for k in ("timestamp", "numInputRows", "inputRowsPerSecond", "durationMs")}, indent=2))

## Step 3 — Stateful streaming: watermark + windowed aggregation + de-duplication

Now we read the bronze table *as a stream* (Delta tables can be both streaming sinks and streaming sources) and compute a windowed aggregate with a watermark, plus de-duplicate by `event_id`. Note: on a first run, `class_streaming_gold_windows` may show few or zero rows -- the `rate` source only has a few seconds of data by the time the query starts, and append-mode windowed aggregation only emits a window once the watermark has advanced past its end. Re-running this cell (which replays from the checkpoint) accumulates enough event time for windows to actually close.

In [ ]:
bronze_stream = spark.readStream.table(table("class_streaming_bronze"))

deduped = (
    bronze_stream
    .withWatermark("event_ts", "2 minutes")
    .dropDuplicates(["event_id"])
)

deduped.createOrReplaceTempView("deduped_bronze_stream")

windowed_revenue = spark.sql("""
SELECT
  window(event_ts, '30 seconds') AS window,
  product_id,
  COUNT(*) AS orders,
  ROUND(SUM(quantity * price), 2) AS revenue
FROM deduped_bronze_stream
WHERE event_type = 'purchase'
GROUP BY
  window(event_ts, '30 seconds'),
  product_id
""")

gold_query = (
    windowed_revenue.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint("class_streaming_gold_windows"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table("class_streaming_gold_windows"))
)
gold_query.awaitTermination()

spark.table(table("class_streaming_gold_windows")).orderBy(F.desc("window.start")).limit(20).show(truncate=False)

## Step 4 — Upserts (MERGE) into a gold table

Some things aren't expressible as a pure streaming DataFrame op -- most commonly, an **upsert (MERGE)** into a dimensional/gold table so re-running or late-arriving corrections update existing rows instead of duplicating them. On an always-on stream you'd typically run this MERGE inside `foreachBatch`, which hands you a regular batch DataFrame per micro-batch; here we get the same result more simply by materializing silver with `availableNow`, then running the MERGE as a plain batch step.

In [ ]:
silver_table = table("class_streaming_silver_deduped")
gold_table = table("class_streaming_gold_upsert")

silver_query = (
    spark.readStream.table(table("class_streaming_bronze"))
    .withWatermark("event_ts", "2 minutes")
    .dropDuplicates(["event_id"])
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint("class_streaming_silver_deduped"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(silver_table)
)
silver_query.awaitTermination()

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_table} (
  window_start TIMESTAMP,
  product_id STRING,
  orders BIGINT,
  revenue DOUBLE
)
USING DELTA
LOCATION '{path("tables", "class_streaming_gold_upsert")}'
""")

spark.sql(f"""
MERGE INTO {gold_table} AS t
USING (
  SELECT
    window(event_ts, '30 seconds').start AS window_start,
    product_id,
    COUNT(*) AS orders,
    ROUND(SUM(quantity * price), 2) AS revenue
  FROM {silver_table}
  WHERE event_type = 'purchase'
  GROUP BY
    window(event_ts, '30 seconds').start,
    product_id
) AS s
ON t.window_start = s.window_start
   AND t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET
  t.orders = s.orders,
  t.revenue = s.revenue
WHEN NOT MATCHED THEN INSERT (
  window_start,
  product_id,
  orders,
  revenue
)
VALUES (
  s.window_start,
  s.product_id,
  s.orders,
  s.revenue
)
""")

spark.table(gold_table).orderBy(F.desc("revenue")).limit(20).show(truncate=False)

## Common streaming pitfalls (talking points)

1. **No watermark on a stateful op** -- Spark keeps state forever; memory grows unbounded until the job dies.
2. **Deleting/moving a checkpoint** -- the stream loses its offset/state history.
3. **Non-deterministic transformations upstream of aggregation** -- makes replay produce different results than the original run.
4. **Output mode mismatch** -- trying `append` mode on an aggregation with no watermark raises an `AnalysisException`.
5. **Assuming exactly-once "just happens"** -- it requires a replayable source, a checkpoint, and an idempotent/transactional sink.

## What's next

The `rate` source is a teaching convenience. Real systems need a durable, replayable, ordered log that many producers and consumers can share -- that's exactly what Kafka/MSK provides, and what `03_kafka_and_msk.ipynb` covers next.